# 02 - Baseline de ML clasico

Pipeline: extraemos features con un backbone preentrenado (ResNet18 frozen)
y entrenamos clasificadores clasicos de scikit-learn.

Etapas:
1. Setup, config y data loaders.
2. Extraccion de features con ResNet18 (sin entrenamiento).
3. Entrenamiento y comparacion de modelos sklearn.
4. Evaluacion en test (accuracy, F1 macro, classification report).
5. Persistencia del mejor modelo y logging opcional a W&B.

In [16]:
load_dotenv(PROJECT_ROOT / ".env")

True

In [7]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from src.utils.config import load_yaml_config
from src.utils.reproducibility import set_global_seed

import joblib
import numpy as np
import torch
from torch import nn
from torchvision import models

from src.utils.wandb_utils import finish_wandb_run, init_wandb_run

CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline_ml.yaml"
config = load_yaml_config(CONFIG_PATH)
set_global_seed(config["seed"])

configured_device = config.get("device", "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else configured_device)
config["runtime_device"] = str(device)

print(f"Configured device (yaml): {configured_device} | Runtime device: {device}")
if torch.cuda.is_available():
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA not available; using CPU.")

config

Configured device (yaml): cpu | Runtime device: cuda
CUDA device name: NVIDIA GeForce RTX 4070 Laptop GPU


{'experiment_name': 'baseline_ml_logreg',
 'seed': 42,
 'device': 'cpu',
 'data': {'root_dir': 'data/asl_alphabet_v1/processed',
  'train_subdir': 'train',
  'val_subdir': 'val',
  'test_subdir': 'test',
  'image_size': 224,
  'batch_size': 32,
  'num_workers': 0},
 'model': {'use_deep_features': True,
  'backbone': 'resnet18',
  'feature_dim': 512},
 'classifier': {'type': 'logistic_regression', 'params': {'max_iter': 500}},
 'tracking': {'use_wandb': True,
  'project': 'signlanguage-classifier',
  'run_name': 'baseline-ml',
  'tags': ['baseline', 'ml']},
 'output': {'artifacts_dir': 'artifacts',
  'model_name': 'baseline_model.joblib'},
 'runtime_device': 'cuda'}

## Data loaders

In [8]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets

from src.data.dataset import build_image_transforms, load_imagefolder_datasets

data_root = PROJECT_ROOT / config["data"]["root_dir"]
image_size = config["data"]["image_size"]
batch_size = config["data"]["batch_size"]
num_workers = config["data"].get("num_workers", 0)

train_dataset, val_dataset = load_imagefolder_datasets(
    root_dir=data_root,
    train_subdir=config["data"].get("train_subdir", "train"),
    val_subdir=config["data"].get("val_subdir", "val"),
    image_size=image_size,
)

test_dir = data_root / config["data"].get("test_subdir", "test")
test_transform = build_image_transforms(image_size=image_size)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

class_names = train_dataset.classes
num_classes = len(class_names)
print({"train": len(train_dataset), "val": len(val_dataset), "test": len(test_dataset), "num_classes": num_classes})


{'train': 156152, 'val': 33461, 'test': 33489, 'num_classes': 29}


## Extraccion de features con ResNet18 (frozen)

In [9]:
from time import perf_counter

from tqdm import tqdm

backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
backbone.fc = nn.Identity()
backbone.eval().to(device)
for p in backbone.parameters():
    p.requires_grad = False

@torch.inference_mode()
def extract_features(loader, split_name: str):
    feats, labels = [], []
    split_t0 = perf_counter()

    progress = tqdm(loader, total=len(loader), desc=f"Extracting {split_name}", unit="batch")
    for images, targets in progress:
        images = images.to(device, non_blocking=True)
        outputs = backbone(images)

        feats.append(outputs.cpu().numpy())
        labels.append(targets.numpy())

    elapsed_s = perf_counter() - split_t0
    print(f"{split_name}: done in {elapsed_s/60:.1f} min ({len(loader.dataset)} images)")
    return np.concatenate(feats), np.concatenate(labels)

X_train, y_train = extract_features(train_loader, "train")
X_val, y_val = extract_features(val_loader, "val")
X_test, y_test = extract_features(test_loader, "test")
print("Feature shape:", X_train.shape)

Extracting train: 100%|██████████| 4880/4880 [30:04<00:00,  2.70batch/s] 


train: done in 30.1 min (156152 images)


Extracting val: 100%|██████████| 1046/1046 [05:56<00:00,  2.93batch/s]


val: done in 5.9 min (33461 images)


Extracting test: 100%|██████████| 1047/1047 [06:18<00:00,  2.77batch/s]

test: done in 6.3 min (33489 images)
Feature shape: (156152, 512)


## Entrenamiento de modelos sklearn

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

candidates = {
    "logreg": LogisticRegression(max_iter=2000, n_jobs=-1, class_weight="balanced"),
    "linear_svc": LinearSVC(class_weight="balanced"),
    "random_forest": RandomForestClassifier(
        n_estimators=200, n_jobs=-1, class_weight="balanced", random_state=config["seed"]
    ),
}

results = {}
for name, model in candidates.items():
    model.fit(X_train_s, y_train)
    val_preds = model.predict(X_val_s)
    results[name] = {
        "model": model,
        "val_accuracy": accuracy_score(y_val, val_preds),
        "val_f1_macro": f1_score(y_val, val_preds, average="macro"),
    }
    print(f"{name}: acc={results[name]['val_accuracy']:.4f} f1_macro={results[name]['val_f1_macro']:.4f}")

logreg: acc=0.9722 f1_macro=0.9730
linear_svc: acc=0.9617 f1_macro=0.9624
random_forest: acc=0.9932 f1_macro=0.9933


## Evaluacion en test del mejor modelo

In [11]:
best_name = max(results, key=lambda k: results[k]["val_f1_macro"])
best = results[best_name]["model"]
print("Best model:", best_name)

test_preds = best.predict(X_test_s)
test_acc = accuracy_score(y_test, test_preds)
test_f1 = f1_score(y_test, test_preds, average="macro")
print("Test accuracy:", test_acc)
print("Test F1 macro:", test_f1)
print(classification_report(y_test, test_preds, target_names=class_names))

Best model: random_forest
Test accuracy: 0.9934306787303293
Test F1 macro: 0.9935985406912768
              precision    recall  f1-score   support

           A       0.99      0.99      0.99      1270
           B       1.00      1.00      1.00      1247
           C       1.00      1.00      1.00      1223
           D       1.00      0.99      0.99      1145
           E       0.99      0.99      0.99      1163
           F       1.00      1.00      1.00      1206
           G       0.99      0.99      0.99      1178
           H       0.99      1.00      0.99      1187
           I       0.99      0.99      0.99      1194
           J       1.00      1.00      1.00      1126
           K       0.99      0.99      0.99      1182
           L       1.00      1.00      1.00      1192
           M       0.99      0.99      0.99      1186
           N       0.99      0.99      0.99      1191
           O       1.00      0.99      1.00      1222
           P       1.00      1.00      1.

## Guardado del modelo y logging W&B (opcional)

In [ ]:
output_dir = PROJECT_ROOT / config["output"]["artifacts_dir"] / "baseline_ml"
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / config["output"]["model_name"]
joblib.dump({"model": best, "scaler": scaler, "class_names": class_names, "best_name": best_name}, model_path)
print("Saved best model to", model_path)

run = init_wandb_run(
    config=config,
    enabled=config["tracking"].get("use_wandb", False),
    project=config["tracking"]["project"],
    run_name=config["tracking"].get("run_name", config["experiment_name"]),
    tags=config["tracking"].get("tags"),
)
if run is not None:
    run.log({
        "best_model": best_name,
        "val_accuracy": results[best_name]["val_accuracy"],
        "val_f1_macro": results[best_name]["val_f1_macro"],
        "test_accuracy": test_acc,
        "test_f1_macro": test_f1,
    })
    finish_wandb_run(run)